In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [2]:
# Config
TARGET_BARANGAYS = ['Nangka', 'Tumana', 'Malanday']

DATA_FILES = [
    '../data/raw/dengue-cases-2022.xlsx',
    '../data/raw/dengue-cases-2023.xlsx',
    '../data/raw/dengue-cases-2024.xlsx',
    '../data/raw/dengue-cases-2025.xlsx',
    '../data/raw/dengue-cases-2026.xlsx',
]

BARANGAY_COL = '(Current Address) Barangay'
DATE_COL     = 'DOnset'

print('Config set.')


Config set.


In [3]:
# Load and combine all files
dfs = []
for f in DATA_FILES:
    df = pd.read_excel(f)
    dfs.append(df)
    print(f'Loaded {f} — {len(df)} rows')

raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(raw)}')

Loaded ../data/raw/dengue-cases-2022.xlsx — 226 rows
Loaded ../data/raw/dengue-cases-2023.xlsx — 2114 rows
Loaded ../data/raw/dengue-cases-2024.xlsx — 1971 rows
Loaded ../data/raw/dengue-cases-2025.xlsx — 1515 rows
Loaded ../data/raw/dengue-cases-2026.xlsx — 254 rows

Total rows: 6080


In [4]:
# Filter for target barangays only
filtered = raw[raw[BARANGAY_COL].isin(TARGET_BARANGAYS)].copy()
filtered = filtered.reset_index(drop=True)

print(f'Filtered rows: {len(filtered)}')
print(filtered[BARANGAY_COL].value_counts())

Filtered rows: 989
(Current Address) Barangay
Nangka      381
Malanday    319
Tumana      289
Name: count, dtype: int64


In [5]:
# Keep only relevant columns
cols_keep = [BARANGAY_COL, DATE_COL, 'Sex', 'AgeYears', 'ClinClass', 'Case Classification', 'outcome']
filtered = filtered[cols_keep]
filtered.columns = ['barangay', 'date_onset', 'sex', 'age', 'clin_class', 'case_class', 'outcome']

print(filtered.head())
print(f'\nDate column dtype: {filtered["date_onset"].dtype}')

  barangay date_onset     sex   age                 clin_class case_class  \
0   Tumana 2022-09-17  Female  15.0  Dengue With Warning Signs  Confirmed   
1   Tumana 2022-09-17  Female  15.0  Dengue With Warning Signs  Confirmed   
2   Nangka 2022-12-23    Male   0.0  Dengue With Warning Signs    Suspect   
3   Nangka 2022-12-30  Female  11.0  Dengue With Warning Signs   Probable   
4   Tumana 2022-12-23    Male   9.0  Dengue With Warning Signs   Probable   

  outcome  
0   Alive  
1   Alive  
2   Alive  
3   Alive  
4   Alive  

Date column dtype: datetime64[us]


In [6]:
# Add week period and aggregate to weekly case counts
filtered['week'] = filtered['date_onset'].dt.to_period('W')

weekly = (
    filtered
    .groupby(['barangay', 'week'])
    .size()
    .reset_index(name='case_count')
)

weekly['week_start'] = weekly['week'].apply(lambda w: w.start_time)
weekly = weekly.sort_values(['barangay', 'week_start']).reset_index(drop=True)

print(f'Weekly records: {len(weekly)}')
print(weekly.head(10))
print('\nSummary per barangay:')
print(weekly.groupby('barangay')['case_count'].describe())

Weekly records: 324
   barangay                   week  case_count week_start
0  Malanday  2022-08-29/2022-09-04           1 2022-08-29
1  Malanday  2022-09-12/2022-09-18           1 2022-09-12
2  Malanday  2023-06-12/2023-06-18           1 2023-06-12
3  Malanday  2023-06-19/2023-06-25           5 2023-06-19
4  Malanday  2023-08-07/2023-08-13           3 2023-08-07
5  Malanday  2023-08-21/2023-08-27           1 2023-08-21
6  Malanday  2023-10-16/2023-10-22           1 2023-10-16
7  Malanday  2023-11-06/2023-11-12           1 2023-11-06
8  Malanday  2023-11-20/2023-11-26           1 2023-11-20
9  Malanday  2023-12-04/2023-12-10           1 2023-12-04

Summary per barangay:
          count      mean       std  min  25%  50%  75%   max
barangay                                                     
Malanday   87.0  3.666667  2.888094  1.0  1.0  3.0  5.0  11.0
Nangka    123.0  3.097561  3.359388  1.0  1.0  2.0  3.0  28.0
Tumana    114.0  2.535088  1.720156  1.0  1.0  2.0  4.0  10.0


In [7]:
# Fill in missing weeks with 0 cases (complete time series)
all_weeks = pd.period_range(
    start='2022-01-01', end='2026-03-31', freq='W'
)

complete_dfs = []
for brgy in TARGET_BARANGAYS:
    df_brgy = pd.DataFrame({'week': all_weeks, 'barangay': brgy})
    df_brgy = df_brgy.merge(
        weekly[weekly['barangay'] == brgy][['week', 'case_count']],
        on='week', how='left'
    )
    df_brgy['case_count'] = df_brgy['case_count'].fillna(0).astype(int)
    df_brgy['week_start'] = df_brgy['week'].apply(lambda w: w.start_time)
    complete_dfs.append(df_brgy)

weekly_complete = pd.concat(complete_dfs).reset_index(drop=True)

print(f'Complete weekly records: {len(weekly_complete)}')
print(weekly_complete.groupby('barangay')['case_count'].describe())

Complete weekly records: 669
          count      mean       std  min  25%  50%  75%   max
barangay                                                     
Malanday  223.0  1.430493  2.538596  0.0  0.0  0.0  2.0  11.0
Nangka    223.0  1.708520  2.930158  0.0  0.0  1.0  2.0  28.0
Tumana    223.0  1.291480  1.768154  0.0  0.0  1.0  2.0  10.0


In [8]:
# Fetch climate data from Open-Meteo API
LAT = 14.6507
LON = 121.1029

url = (
    f'https://archive-api.open-meteo.com/v1/archive'
    f'?latitude={LAT}&longitude={LON}'
    f'&start_date=2022-01-01&end_date=2026-03-31'
    f'&daily=precipitation_sum,temperature_2m_mean,relative_humidity_2m_mean'
    f'&timezone=Asia/Manila'
)

print('Fetching climate data...')
resp = requests.get(url)
data = resp.json()

climate = pd.DataFrame({
    'date':     pd.to_datetime(data['daily']['time']),
    'rainfall': data['daily']['precipitation_sum'],
    'temp':     data['daily']['temperature_2m_mean'],
    'humidity': data['daily']['relative_humidity_2m_mean'],
})

print(f'Climate rows: {len(climate)}')
print(climate.head())

Fetching climate data...
Climate rows: 1551
        date  rainfall  temp  humidity
0 2022-01-01       0.0  24.1        77
1 2022-01-02       0.0  23.5        77
2 2022-01-03       0.0  24.0        74
3 2022-01-04       0.0  24.5        72
4 2022-01-05       0.0  24.7        74


In [9]:
# Aggregate climate to weekly
climate['week'] = climate['date'].dt.to_period('W')

climate_weekly = (
    climate
    .groupby('week')
    .agg(
        rainfall=('rainfall', 'sum'),
        temp=('temp', 'mean'),
        humidity=('humidity', 'mean')
    )
    .reset_index()
)

climate_weekly['week_start'] = climate_weekly['week'].apply(lambda w: w.start_time)

print(f'Weekly climate rows: {len(climate_weekly)}')
print(climate_weekly.head())

Weekly climate rows: 223
                    week  rainfall       temp   humidity week_start
0  2021-12-27/2022-01-02       0.0  23.800000  77.000000 2021-12-27
1  2022-01-03/2022-01-09       7.6  24.557143  76.714286 2022-01-03
2  2022-01-10/2022-01-16       3.4  24.371429  76.000000 2022-01-10
3  2022-01-17/2022-01-23      18.1  24.957143  76.285714 2022-01-17
4  2022-01-24/2022-01-30      27.6  26.242857  79.571429 2022-01-24


In [10]:
# Merge cases with climate data
merged = weekly_complete.merge(
    climate_weekly[['week', 'rainfall', 'temp', 'humidity']],
    on='week', how='left'
)

# Apply 4-week lag per barangay
lag_weeks = 4
lagged_dfs = []

for brgy in TARGET_BARANGAYS:
    df_brgy = merged[merged['barangay'] == brgy].copy().sort_values('week_start')
    df_brgy['rainfall_lag'] = df_brgy['rainfall'].shift(lag_weeks)
    df_brgy['temp_lag']     = df_brgy['temp'].shift(lag_weeks)
    df_brgy['humidity_lag'] = df_brgy['humidity'].shift(lag_weeks)
    lagged_dfs.append(df_brgy)

final = pd.concat(lagged_dfs).dropna().reset_index(drop=True)

print(f'Final dataset rows: {len(final)}')
print(final.head())

Final dataset rows: 657
                    week barangay  case_count week_start  rainfall       temp  \
0  2022-01-24/2022-01-30   Nangka           0 2022-01-24      27.6  26.242857   
1  2022-01-31/2022-02-06   Nangka           0 2022-01-31       2.7  25.671429   
2  2022-02-07/2022-02-13   Nangka           0 2022-02-07      20.3  26.342857   
3  2022-02-14/2022-02-20   Nangka           0 2022-02-14      27.6  26.385714   
4  2022-02-21/2022-02-27   Nangka           0 2022-02-21      15.5  26.614286   

    humidity  rainfall_lag   temp_lag  humidity_lag  
0  79.571429           0.0  23.800000     77.000000  
1  73.428571           7.6  24.557143     76.714286  
2  74.714286           3.4  24.371429     76.000000  
3  77.428571          18.1  24.957143     76.285714  
4  74.285714          27.6  26.242857     79.571429  


In [11]:
# Add outbreak label for Logistic Regression
# Threshold: weeks with case_count >= 75th percentile per barangay = outbreak
threshold = final.groupby('barangay')['case_count'].transform(lambda x: x.quantile(0.75))
final['outbreak'] = (final['case_count'] >= threshold).astype(int)

print('Outbreak distribution:')
print(final['outbreak'].value_counts())
print('\nPer barangay:')
print(final.groupby('barangay')['outbreak'].value_counts())

Outbreak distribution:
outbreak
0    442
1    215
Name: count, dtype: int64

Per barangay:
barangay  outbreak
Malanday  0           159
          1            60
Nangka    0           134
          1            85
Tumana    0           149
          1            70
Name: count, dtype: int64


In [12]:
# Select final feature columns
feature_cols = ['barangay', 'week_start', 'case_count', 'outbreak',
                'rainfall_lag', 'temp_lag', 'humidity_lag']

export = final[feature_cols].copy()

# Export to CSV
export.to_csv('../data/processed_dataset.csv', index=False)

print('Exported to data/processed_dataset.csv')
print(f'Shape: {export.shape}')
print(export.head(10))

Exported to data/processed_dataset.csv
Shape: (657, 7)
  barangay week_start  case_count  outbreak  rainfall_lag   temp_lag  \
0   Nangka 2022-01-24           0         0           0.0  23.800000   
1   Nangka 2022-01-31           0         0           7.6  24.557143   
2   Nangka 2022-02-07           0         0           3.4  24.371429   
3   Nangka 2022-02-14           0         0          18.1  24.957143   
4   Nangka 2022-02-21           0         0          27.6  26.242857   
5   Nangka 2022-02-28           0         0           2.7  25.671429   
6   Nangka 2022-03-07           0         0          20.3  26.342857   
7   Nangka 2022-03-14           0         0          27.6  26.385714   
8   Nangka 2022-03-21           0         0          15.5  26.614286   
9   Nangka 2022-03-28           0         0           2.7  27.414286   

   humidity_lag  
0     77.000000  
1     76.714286  
2     76.000000  
3     76.285714  
4     79.571429  
5     73.428571  
6     74.714286  
7     77

In [1]:
import pandas as pd
df = pd.read_csv('../data/processed_dataset.csv')
df['week_start'] = pd.to_datetime(df['week_start'])

print('=== DATE RANGE ===')
print(f'From: {df["week_start"].min()}')
print(f'To:   {df["week_start"].max()}')

print('\n=== ROWS PER BARANGAY ===')
print(df.groupby('barangay').size())

print('\n=== ROWS PER YEAR ===')
df['year'] = df['week_start'].dt.year
print(df.groupby(['barangay','year']).size().unstack())

print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

print('\n=== SAMPLE ===')
print(df.head(10))

=== DATE RANGE ===
From: 2022-01-24 00:00:00
To:   2026-03-30 00:00:00

=== ROWS PER BARANGAY ===
barangay
Malanday    219
Nangka      219
Tumana      219
dtype: int64

=== ROWS PER YEAR ===
year      2022  2023  2024  2025  2026
barangay                              
Malanday    49    52    53    52    13
Nangka      49    52    53    52    13
Tumana      49    52    53    52    13

=== MISSING VALUES ===
barangay        0
week_start      0
case_count      0
outbreak        0
rainfall_lag    0
temp_lag        0
humidity_lag    0
year            0
dtype: int64

=== SAMPLE ===
  barangay week_start  case_count  outbreak  rainfall_lag   temp_lag  \
0   Nangka 2022-01-24           0         0           0.0  23.800000   
1   Nangka 2022-01-31           0         0           7.6  24.557143   
2   Nangka 2022-02-07           0         0           3.4  24.371429   
3   Nangka 2022-02-14           0         0          18.1  24.957143   
4   Nangka 2022-02-21           0         0          27.6

In [2]:
# Check if all barangays have the same weeks
nangka   = set(df[df['barangay']=='Nangka']['week_start'])
tumana   = set(df[df['barangay']=='Tumana']['week_start'])
malanday = set(df[df['barangay']=='Malanday']['week_start'])

print('Nangka == Tumana:', nangka == tumana)
print('Nangka == Malanday:', nangka == malanday)

# Check zero case weeks per barangay
print('\n=== ZERO CASE WEEKS ===')
print(df[df['case_count']==0].groupby('barangay').size())

print('\n=== NON-ZERO CASE WEEKS ===')
print(df[df['case_count']>0].groupby('barangay').size())

# Check actual case totals
print('\n=== TOTAL CASES PER BARANGAY ===')
print(df.groupby('barangay')['case_count'].sum())

print('\n=== TOP 10 HIGHEST WEEKS ===')
print(df.nlargest(10, 'case_count')[['barangay','week_start','case_count']])

Nangka == Tumana: True
Nangka == Malanday: True

=== ZERO CASE WEEKS ===
barangay
Malanday    132
Nangka       96
Tumana      106
dtype: int64

=== NON-ZERO CASE WEEKS ===
barangay
Malanday     87
Nangka      123
Tumana      113
dtype: int64

=== TOTAL CASES PER BARANGAY ===
barangay
Malanday    319
Nangka      381
Tumana      288
Name: case_count, dtype: int64

=== TOP 10 HIGHEST WEEKS ===
     barangay week_start  case_count
88     Nangka 2023-10-02          28
71     Nangka 2023-06-05          17
102    Nangka 2024-01-08          11
567  Malanday 2024-07-15          11
575  Malanday 2024-09-09          11
576  Malanday 2024-09-16          11
579  Malanday 2024-10-07          11
586  Malanday 2024-11-25          11
86     Nangka 2023-09-18          10
90     Nangka 2023-10-16          10


In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(df.to_string())

     barangay week_start  case_count  outbreak  rainfall_lag   temp_lag  humidity_lag  year
0      Nangka 2022-01-24           0         0           0.0  23.800000     77.000000  2022
1      Nangka 2022-01-31           0         0           7.6  24.557143     76.714286  2022
2      Nangka 2022-02-07           0         0           3.4  24.371429     76.000000  2022
3      Nangka 2022-02-14           0         0          18.1  24.957143     76.285714  2022
4      Nangka 2022-02-21           0         0          27.6  26.242857     79.571429  2022
5      Nangka 2022-02-28           0         0           2.7  25.671429     73.428571  2022
6      Nangka 2022-03-07           0         0          20.3  26.342857     74.714286  2022
7      Nangka 2022-03-14           0         0          27.6  26.385714     77.428571  2022
8      Nangka 2022-03-21           0         0          15.5  26.614286     74.285714  2022
9      Nangka 2022-03-28           0         0           2.7  27.414286     68.4